# HeatUp 02 - stability analysis (four gates)

| gate | question | evidence |
|---|---|---|
| **1** mechanical | positive-definite C (Born-Huang)? | relaxed-ion elastic tensor |
| **0** structural transition | did the structure stay in its basin? | block-averaged positions / cell / symmetry breaking from the MD |
| **2** vibrational | soft modes? | MD VDOS (momenta VACF) and harmonic phonons |
| **3** thermodynamic | above the T-dependent hull of its competitors? | G(T) with the *same* method for every phase |

Gates are **independent**: `cfg.gates.enabled` selects which ones run, every enabled gate is evaluated whatever
the others say (a Born-unstable structure still gets its MD and hull verdicts), and the steps a gate needs are run
only if that gate is enabled. Verdicts are always recomputed from cached step outputs, so changing a threshold
never needs a rerun; `cfg.run.force` forces step recomputation.

## Where the data lives
```
<database_root>/                    targets: read + write
└── <material>/<label>/             e.g. BaTiO3/P4mm
    ├── POSCAR                      the input structure
    ├── relaxation/   CONTCAR, energy.json, run.traj          step "relax"
    ├── elastic/      elastic.json                            step "elastic"   -> Gate 1
    ├── phonons/      ha.json, [qha.json], phonon.png         steps "phonons", "qha"
    ├── md/<T>K/      output.traj, ...                        step "md"        -> Gates 0, 2
    └── stability/    report_<stage>.json, result_<stage>.json, report_<stage>.png, zero_K_hull.json
<candidates_root>/                  user inputs: read-only, never written
<competing_root>/                   competing phases (MP downloads, generated, mirrored): same <material>/<label>/ layout,
                                    each with its own relaxation/ and phonons/ (they feed Gate 3)
```
Every step directory carries a `manifest.json` (config fingerprint + input hashes): a step is recomputed when its
inputs or relevant options changed, or when it is listed in `cfg.run.force`.

In [ ]:
import os, warnings
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import heatup
from heatup import cfg, run_step, PhaseDir, override
from heatup.runtime import verbose
from heatup.report import print_layout

warnings.filterwarnings("ignore")
verbose()
print("HeatUp", heatup.__version__)
from heatup import (assess_mechanical_stability, assess_structural_transition, assess_vibrational_stability,
                    assess_thermodynamic_stability, run_pipeline, screen, ScreenOptions)
from heatup.report import gate_table, style_status, show_stability_figure, plot_free_energy_terms, plot_vdos

## Configuration

In [ ]:
# --- calculator -----------------------------------------------------------
cfg.calc.backend = os.environ.get("HEATUP_BACKEND", "mace-mp")
cfg.calc.model = os.environ.get("MACE_MODEL_PATH", "medium-mpa-0")      # mace_mp name or path to a .model
cfg.calc.model_overrides = {}                                           # e.g. {"BaTiO3": "/path/BTO_MACE.model"}
DEVICE = None                                                           # None -> cuda if usable, else cpu

cfg.phases.database_root = "database"
cfg.phases.candidates_root = "input/candidates"          # read-only user structures
cfg.phases.competing_root = "competing_phases"           # MP downloads / generated / mirrored competing phases + their results
cfg.phases.sources = ["database", "candidates", "competing", "mp-api"]     # mp-api needs cfg.phases.mp_api_key or $MP_API_KEY

cfg.hull.operating_T = 600.0
cfg.hull.temperatures = list(range(0, 1501, 50))
cfg.hull.phonon_method = "HA"                            # "HA" | "QHA" | "VDOS"
cfg.hull.target_overlay = "VDOS"                         # anharmonic estimate reported next to the consistent hull
cfg.hull.prune_window_eV = 0.15                          # competitors above the 0 K hull by more get no phonons

cfg.gates.enabled = ["mechanical", "structural_transition", "vibrational", "thermodynamic"]   # drop any you do not want
cfg.run.force = []                                       # e.g. ["md"] or "all": recompute those steps whatever the cache says
cfg.gates.mobile_species = []                            # e.g. ["Li"] for a superionic conductor
cfg.gates.hull_ok_eV   = 0.010
cfg.gates.hull_fail_eV = 0.100

TARGETS = ["Cu/Fm-3m"]  # database/<material>/<label>
MD_TEMPERATURES = [600]

## Gate by gate
Cheapest first. Each gate only needs its own steps, and none stops the others.

In [ ]:
ENABLED = cfg.gates.enabled
results = {}
for t in TARGETS:
    d = PhaseDir(os.path.join(cfg.phases.database_root, t))
    print("=" * 70, "\n", d.tag, "| gates:", ", ".join(ENABLED))
    g = results[t] = {}

    run_step("relax", d, device=DEVICE)
    if "mechanical" in ENABLED:
        run_step("elastic", d, device=DEVICE)
        g["mechanical"] = assess_mechanical_stability(d)
    if "vibrational" in ENABLED:
        run_step("phonons", d, device=DEVICE)
    if "structural_transition" in ENABLED or "vibrational" in ENABLED:
        for T in MD_TEMPERATURES:
            run_step("md", d, device=DEVICE, temperature=T)
    if "structural_transition" in ENABLED:
        g["structural_transition"] = assess_structural_transition(d)
    if "vibrational" in ENABLED:
        g["vibrational"] = assess_vibrational_stability(d)
    if "thermodynamic" in ENABLED:                       # discovers competitors, relaxes them, phonons, hull
        g["thermodynamic"] = assess_thermodynamic_stability(d, device=DEVICE)

    for name, r in g.items():
        print(f"  {name:22s} {r['status']:8s} {r['message']}")

## What the run wrote
Targets and competing phases live in separate trees; the competitors carry their own results.

In [ ]:
print_layout(cfg.phases.database_root, max_depth=4)
print()
print_layout(cfg.phases.competing_root, max_depth=3)

## Assemble the reports (reads the cache only) and look at them

In [ ]:
reports = {t: run_pipeline(os.path.join(cfg.phases.database_root, t), compute=False,
                           md_temperatures=MD_TEMPERATURES) for t in TARGETS}
display(style_status(gate_table(reports)))
for t, r in reports.items():
    display(show_stability_figure(r))
    for f in r["flags"]:
        print(f)

## Hull in detail: consistent hull vs anharmonic overlay
The consistent curve compares like with like (every phase with the same method). The overlay adds
`F_VDOS - F_HA` of the target only: an *estimate* of the anharmonic stabilisation, not a hull.

In [ ]:
for t, r in reports.items():
    plot_free_energy_terms(r)
    plot_vdos(r)
    plt.show()
    print("decomposition at operating T:", (r["thermodynamic"] or {}).get("decomposition"))

## Running a subset of gates
Same call, different config: only the listed gates are computed and reported; the others show as `skipped`.

In [ ]:
subset = {t: run_pipeline(os.path.join(cfg.phases.database_root, t), gates=["mechanical", "thermodynamic"],
                             compute=False, save_figure=False, md_temperatures=MD_TEMPERATURES) for t in TARGETS}
for t, r in subset.items():
    print(t, "| enabled:", r["enabled"], "| skipped:", r["skipped"], "| overall:", r["overall"])

## Thresholds can change without recomputing

In [ ]:
from heatup import override
with override(**{"gates.hull_ok_eV": 0.002, "gates.zero_frac_warn": 0.002}):
    strict = {t: run_pipeline(os.path.join(cfg.phases.database_root, t), compute=False, save_figure=False)
              for t in TARGETS}
display(style_status(gate_table(strict)))

## The same thing in one call (what other software uses)
`screen` returns a typed verdict; `stage="pre_md"` runs relax + (enabled) mechanical + 0 K hull, no MD, `stage="post_md"` runs everything.

In [ ]:
for t in TARGETS:
    for stage in ("pre_md", "post_md"):
        r = screen(os.path.join(cfg.phases.database_root, t), stage=stage,
                   options=ScreenOptions(operating_T=cfg.hull.operating_T, md_temperatures=MD_TEMPERATURES, device=DEVICE,
                                 gates=cfg.gates.enabled, force=None))
        print(f"{t:12s} {stage:8s} {r.verdict:12s} E_hull = {r.e_hull_eV}  {r.reasons}")